In [ ]:
import os
import random
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split
from torchmetrics.functional import accuracy as tm_accuracy
from torchmetrics.functional import f1_score as tm_f1_score
from itertools import product

from src.dfg_ms_plexus.data import MRIDataset
from src.dfg_ms_plexus.model import ResNet3DMRI
from src.dfg_ms_plexus.training_setup import get_labels_full, get_labels_hc_ms, get_labels_hc_cis_ms
from src.dfg_ms_plexus.evaluation import make_classification_report_tables

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    #torch.backends.cudnn.deterministic = True
    #torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def infinite_loader(loader):
    while True:
        for batch in loader:   # reshuffles each pass if shuffle=True
            yield batch

def initialize_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.xavier_uniform_(module.weight)
    elif isinstance(module, nn.Conv3d):
        torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')

In [ ]:
STEPS = 5000  # 10000
BATCH_SIZE = 8  # 32
LR = 1e-4
CLIP_GRAD_NORM = 1.0
WEIGHT_DECAY = 1e-2
BASE_FILTERS = 8
DROPOUT = 0.3  # 0.2
VAL_FREQ = 100 # 200
NORM_PER_VOL = True
NORMALIZATION = nn.InstanceNorm3d
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULT_DIR = Path("results/cnn_clf_harmonized/")
RESULT_DIR.mkdir(exist_ok=True)

ROOT = Path('/media/yannik/Intenso/DATA/dfg_plexus')
HC_MRI_DIR = ROOT / "HC_T1s___freesurfer___reorientated_padded___harmonized___cropped/"
MS_MRI_DIR = ROOT / "T1s___float32___harmonized___cropped/"

In [ ]:
full_train_ds = MRIDataset(
    hc_mri_dir=HC_MRI_DIR,
    ms_mri_dir=MS_MRI_DIR,
    class_mapping=get_labels_hc_cis_ms,
    # sample_ids=root / 'splits/train_idx_SA.pkl',
    sample_ids=ROOT / 'splits_harmonized/train_idx_SA.pkl',
    normalize_per_volume=NORM_PER_VOL,
)

test_ds = MRIDataset(
    hc_mri_dir=HC_MRI_DIR,
    ms_mri_dir=MS_MRI_DIR,
    class_mapping=get_labels_hc_cis_ms,
    # sample_ids=root / 'splits/test_idx_SA.pkl',
    sample_ids=ROOT / 'splits_harmonized/test_idx_SA.pkl',
    normalize_per_volume=NORM_PER_VOL,
)

test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

TASK, N_CLASSES = 'multiclass', 3
CLASS_ORDER = [0, 1, 2]
CLASS_NAMES = ["hc", "CIS", "MS"]

In [ ]:
def make_loss_fn():
    """Create the weighted cross-entropy loss used for this classification task."""
    class_counts = torch.tensor([21, 14, 125], dtype=torch.float32, device=DEVICE)
    # class_weights = 1.0 / class_counts  # Inverse frequency
    # class_weights = 1.0 / class_counts**2  # Inverse squared
    class_weights = 1.0 / torch.sqrt(class_counts)
    class_weights = class_weights / class_weights.sum()

    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

In [ ]:
def evaluate_model(model, dataloader, loss_fn=None):
    model.eval()

    losses = []
    y_true = None
    y_pred = None

    with torch.no_grad():
        for sample, label in dataloader:
            sample = sample.to(DEVICE)
            label = label.to(DEVICE).long().view(-1)

            out = model(sample)
            preds = out.argmax(-1)

            if loss_fn is not None:
                losses.append(loss_fn(out, label).item())

            y_true = label if y_true is None else torch.cat([y_true, label])
            y_pred = preds if y_pred is None else torch.cat([y_pred, preds])

    macro_eval_acc = tm_accuracy(
        y_pred,
        y_true,
        task=TASK,
        num_classes=N_CLASSES,
        average='macro',
    )

    macro_eval_f1 = tm_f1_score(
        y_pred,
        y_true,
        task=TASK,
        num_classes=N_CLASSES,
        average='macro',
    )

    return {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "balanced_accuracy": macro_eval_acc,
        "macro_f1": macro_eval_f1,
        "y_true": y_true.detach().cpu().numpy(),
        "y_pred": y_pred.detach().cpu().numpy(),
    }

In [ ]:
def run_one_seed_one_fold(seed: int, fold: int):
    print(f"\n\n### Seed {seed} Fold {fold} ###")

    seed_everything(seed)

    train_ds = MRIDataset(
        hc_mri_dir=HC_MRI_DIR,
        ms_mri_dir=MS_MRI_DIR,
        class_mapping=get_labels_hc_cis_ms,
        # sample_ids=root / f'splits/train_idx_seed_{seed}_fold_{fold}.pkl',
        sample_ids=root / f'splits_harmonized/train_idx_seed_{seed}_fold_{fold}.pkl',
        normalize_per_volume=NORM_PER_VOL,
    )

    val_ds = MRIDataset(
        hc_mri_dir=HC_MRI_DIR,
        ms_mri_dir=MS_MRI_DIR,
        class_mapping=get_labels_hc_cis_ms,
        # sample_ids=root / f'splits/val_idx_seed_{seed}_fold_{fold}.pkl',
        sample_ids=root / f'splits_harmonized/val_idx_seed_{seed}_fold_{fold}.pkl',
        normalize_per_volume=NORM_PER_VOL,
    )

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    m = ResNet3DMRI(
        num_classes=N_CLASSES,
        base_filters=BASE_FILTERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optim = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ConstantLR(optim)

    loss_fn = make_loss_fn()

    train_iter = infinite_loader(train_dl)

    per_step_loss = []
    per_step_acc = []

    best_val_f1 = -np.inf
    best_val_loss = np.inf

    for step in range(STEPS):
        m.train()

        sample, label = next(train_iter)
        sample = sample.cuda()
        label = label.cuda().squeeze()

        optim.zero_grad()

        out = m(sample)
        loss = loss_fn(out, label)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), CLIP_GRAD_NORM)
        optim.step()
        sched.step()

        per_step_loss.append(loss.item())

        macro_acc = tm_accuracy(
            out.argmax(-1),
            label,
            task=TASK,
            num_classes=N_CLASSES,
            average='macro',
        )
        per_step_acc.append(macro_acc.item())

        if step % VAL_FREQ == 0:

            val_results = evaluate_model(m, val_dl, loss_fn)

            mean_train_loss = np.mean(per_step_loss)
            mean_train_acc = np.mean(per_step_acc)

            per_step_loss = []
            per_step_acc = []

            mean_val_loss = val_results["loss"]
            mean_val_acc = val_results["balanced_accuracy"]
            mean_val_f1 = val_results["macro_f1"]

            print(
                f"Step {step}"
                f"\t || Train loss: {mean_train_loss:.3f}"
                f"\tTrain acc: {mean_train_acc:.3f}"
                f"\t || Val loss: {mean_val_loss:.3f}"
                f"\tVal acc: {mean_val_acc:.3f}"
                f"\tVal F1: {mean_val_f1:.3f}"
            )

            if mean_val_f1 > best_val_f1:
                best_val_f1 = mean_val_f1
                best_val_f1_path = RESULT_DIR / f"cnn_clf___seed_{seed}___fold_{fold}___best_val_f1.pth"
                torch.save(m.state_dict(), best_val_f1_path)

            if mean_val_loss < best_val_loss:
                best_val_loss = mean_val_loss
                best_val_loss_path = RESULT_DIR / f"cnn_clf___seed_{seed}___fold_{fold}___best_val_loss.pth"
                torch.save(m.state_dict(), best_val_loss_path)


    return {
        "seed": seed,
        "fold": fold,
        "best_val_f1": float(best_val_f1),
        "best_val_loss": float(best_val_loss),
        "best_val_f1_path": str(best_val_f1_path),
        "best_val_loss_path": str(best_val_loss_path),
    }

In [ ]:
# Training on five different seeds on five folds

SEEDS = [0, 1, 2, 3, 4]
FOLDS = [0, 1, 2, 3, 4]

results = []

for seed, fold in product(SEEDS, FOLDS):
    result = run_one_seed_one_fold(seed, fold)
    results.append(result)

cv_results_df = pd.DataFrame(results)
cv_results_df.to_csv(RESULT_DIR / "cnn_clf___multiseed___cross_validation_results.csv", index=False)
cv_results_df

In [ ]:
import shutil

# Selecting the best model for each seed

# MODEL_SELECTION_METRIC = "best_val_f1"
MODEL_SELECTION_METRIC = "best_val_loss"

cv_results_df = pd.read_csv(RESULT_DIR / "cnn_clf___multiseed___cross_validation_results.csv")

best_models_per_seed_df = (
    cv_results_df
    .sort_values(
        by=["seed", MODEL_SELECTION_METRIC, "best_val_loss"],
        ascending=[True, False, True],
    )
    .groupby("seed", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

selected_checkpoint_paths = []

for _, row in best_models_per_seed_df.iterrows():
    seed = int(row["seed"])

    source_checkpoint_path = Path(row[f"{MODEL_SELECTION_METRIC}_path"])

    target_checkpoint_path = RESULT_DIR / f"cnn_clf___seed_{seed}___{MODEL_SELECTION_METRIC}.pth"

    if not source_checkpoint_path.exists():
        raise FileNotFoundError(
            f"Could not find selected checkpoint for seed {seed}: "
            f"{source_checkpoint_path}"
        )

    shutil.copy2(source_checkpoint_path, target_checkpoint_path)

    selected_checkpoint_paths.append(str(target_checkpoint_path))

best_models_per_seed_df["selected_checkpoint_path"] = selected_checkpoint_paths

best_models_per_seed_df

In [ ]:
# Evaluate best model per seed on test set

MODEL_SELECTION_METRIC = "best_val_loss"

test_results = []

test_loss_fn = make_loss_fn()

for _, row in best_models_per_seed_df.iterrows():
    seed = int(row["seed"])
    fold = int(row["fold"])
    checkpoint_path = Path(row[f"{MODEL_SELECTION_METRIC}_path"])

    m = ResNet3DMRI(
        num_classes=N_CLASSES,
        base_filters=BASE_FILTERS,
        dropout=DROPOUT,
    ).to(DEVICE)
    m.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))

    metrics = evaluate_model(m, test_dl, loss_fn=test_loss_fn)

    test_results.append({
        "seed": seed,
        "selected_fold": fold,
        "checkpoint": str(checkpoint_path),
        "cv_best_val_f1": float(row["best_val_f1"]),
        "cv_best_val_loss": float(row["best_val_loss"]),

        # Required by classification_report_summary.py
        "labels": metrics["y_true"],
        "predictions": metrics["y_pred"],
    })


In [ ]:
tables = make_classification_report_tables(
    test_results,
    class_order=CLASS_ORDER,
    class_names=CLASS_NAMES,
    metadata_keys=[
        "seed",
        "selected_fold",
        "cv_best_val_f1",
        "cv_best_val_loss",
        "test_loss",
        "test_balanced_accuracy",
    ],
    decimals=3,
)

print("\nSeed-level test results:")
display(tables.seed_level_results)

print("\nPer-seed sklearn-style classification reports:")
display(tables.per_seed_report)

print("\nClassification report summary on the fixed test set, mean ± std across seeds:")
display(tables.formatted_summary)